# 第 1 天练习 —— 基于熵的市场结构分析（NVIDIA Nemotron）

## 练习目标（理念）

用 **Binance** 公开 K 线数据计算 **Shannon 熵** 与 **Gaussian 熵**，再把指标交给 **OpenRouter** 上的 NVIDIA 推理模型做市场结构解读：

- **输入**：BTCUSDT 1 分钟蜡烛 → 对数收益 → 熵指标
- **输出**：量化指标 + 模型对 regime（趋势 / 震荡 / 压缩等）的结构化解释
- **模型**：`nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free`（经 OpenRouter）

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| HTTP / API 调用 | `requests.post` → OpenRouter Chat Completions |
| `messages`（system / user） | system 定分析角色，user 塞入熵数值 |
| 环境变量 | `.env` 里的 `OPENROUTER_API_KEY` |
| 数据预处理 | pandas / numpy 把原始 kline 变成可算熵的分布 |

## 怎么跑

1. 安装依赖：`python-binance`、`numpy`、`pandas`、`scipy`、`openai`、`python-dotenv`、`requests` 等
2. 准备 `.env`：写入 `OPENROUTER_API_KEY`
3. 从上到下依次运行：先拉 K 线算熵，再调模型解读


In [29]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 math：手写 Shannon 熵时用 log2，以及高斯熵公式里的 pi、e
import math
# 从 python-binance 导入 Client：无需密钥也能拉公开市场的 K 线（klines）
from binance.client import Client
# 导入 numpy：直方图、对数收益、数组运算
import numpy as np
# 导入 pandas：把币安返回的列表整理成 DataFrame，方便列操作
import pandas as pd
# 从 scipy.stats 导入 entropy：用库函数交叉验证手写 Shannon 熵
from scipy.stats import entropy
# 导入 requests：用 HTTP POST 直接调 OpenRouter 的 chat/completions
import requests
# 导入 json：把请求体序列化成 JSON 字符串（本练习用 data=json.dumps(...)）
import json
# 导入标准库 os：从环境变量读取 API Key
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：后面用 OpenRouter 兼容的 base_url 初始化（本格先导入备用）
from openai import OpenAI


## 拉币安 K 线（无需 API Key）

下面用 `python-binance` 的公开接口拉取 **BTCUSDT** 的 **1 分钟**蜡烛数据。  
公开行情接口一般**不需要**币安 API 密钥；只要网络能访问 Binance 即可。


In [30]:
# ========== 币安公开客户端 + 拉取 1 分钟 K 线 ==========

# 创建币安 Client；无参时走公开行情，不必填 api_key / api_secret
client = Client()

# get_klines：拉取 K 线列表；每根蜡烛是「开高低收量」等字段的数组
klines = client.get_klines(
    # 交易对：比特币对 USDT（字符串必须与币安符号一致）
    symbol='BTCUSDT',
    # 时间粒度：1 分钟；常量来自 Client，避免手写错 interval 字符串
    interval=Client.KLINE_INTERVAL_1MINUTE,
    # 最多取最近 500 根，控制后面直方图样本量
    limit=500
)


In [31]:
# ========== 把原始 klines 列表转成带列名的 DataFrame ==========

# DataFrame：二维表；columns 顺序必须与币安 kline 字段顺序一致，否则列会对错位
df = pd.DataFrame(klines, columns=[
    'open_time', 'open', 'high', 'low', 'close', 'volume',
    'close_time', 'quote_asset_volume', 'num_trades',
    'taker_buy_base', 'taker_buy_quote', 'ignore'
])
# 打印预览：确认行数、列名、数值是否像价格/成交量
print(df)


         open_time            open            high             low  \
0    1778309040000  80238.57000000  80238.58000000  80222.13000000   
1    1778309100000  80222.14000000  80222.14000000  80208.33000000   
2    1778309160000  80217.35000000  80219.65000000  80210.35000000   
3    1778309220000  80219.65000000  80281.13000000  80219.64000000   
4    1778309280000  80261.53000000  80270.00000000  80253.62000000   
..             ...             ...             ...             ...   
495  1778338740000  80332.23000000  80332.23000000  80332.22000000   
496  1778338800000  80332.23000000  80332.23000000  80321.23000000   
497  1778338860000  80321.23000000  80330.21000000  80320.00000000   
498  1778338920000  80330.21000000  80349.40000000  80330.20000000   
499  1778338980000  80349.40000000  80349.40000000  80349.39000000   

              close       volume     close_time quote_asset_volume  \
0    80222.14000000   3.32787000  1778309099999    266987.89462700   
1    80217.35000000

In [32]:
# ========== 从收盘价算对数收益 → Shannon / Gaussian 熵 ==========

# 币安返回的价格常是字符串；先转 float 才能做算术
df['close'] = df['close'].astype(float)

# 对数收益（log returns）：ln(P_t / P_{t-1})；比简单百分比收益更适合信息论度量
df['returns'] = np.log(df['close'] / df['close'].shift(1))

# 第一行没有前一期价格，会产生 NaN；丢掉后再做直方图
returns = df['returns'].dropna()

# 把连续收益切成 20 个箱子（bins），得到各箱计数 hist
hist, bins = np.histogram(returns, bins=20)

# 计数 → 概率：除以总和，使概率质量和为 1
probabilities = hist / hist.sum()

# 去掉概率为 0 的箱子：log(0) 无定义，会破坏熵求和
probabilities = probabilities[probabilities > 0]

# 手写 Shannon 熵：H = -Σ p log2(p)；单位是 bit（因为 log2）
shannon_entropy = -sum(
    p * math.log2(p)
    for p in probabilities
)

# 用 scipy.stats.entropy 交叉验证；base=2 与手写 log2 一致
h=entropy(probabilities,base=2)

# 打印两种 Shannon 熵，数值应几乎相同（浮点误差可忽略）
print("Shannon Entropy:", shannon_entropy)
print("Shannon Entropy using scipy:", h)

# 再取一列收盘价（float），供后面方差 / 高斯熵使用
close = df["close"].astype(float)

# 再次计算对数收益并去 NaN（与上面逻辑等价，变量名 returns 会被覆盖）
returns = np.log(close / close.shift(1)).dropna()

# 收益序列的方差 σ²：高斯熵公式的核心输入
variance = np.var(returns)

# 高斯熵（连续正态近似）：H = 1/2 * log2(2πe σ²)
G = 0.5 * math.log2(
    2 * math.pi * math.e * variance
)

# 打印高斯熵；负值常见于方差很小（波动压缩）时
print("Gaussian Entropy:", G)


Shannon Entropy: 2.752202013798435
Shannon Entropy using scipy: 2.7522020137984344
Gaussian Entropy: -10.242077636891816


## 把熵交给 NVIDIA 推理模型解读

上面已经用 **手写公式** 和 **`scipy.stats.entropy`** 算出了 Shannon 熵，并用方差得到了 Gaussian 熵。

下一步：把这两个标量指标放进 **user prompt**，让 OpenRouter 上的 Nemotron 推理模型做市场结构（regime）解读——趋势、震荡、波动扩张/压缩等。


In [33]:
# ========== 环境变量 + OpenRouter 兼容的 OpenAI 客户端 ==========

# 加载 .env；override=True 表示用文件里的值覆盖进程里已有同名环境变量
load_dotenv(override=True)
# 从环境变量读取 OpenRouter 密钥（不要把真实 key 写进笔记本）
api_key=os.getenv("OPENROUTER_API_KEY")
# 创建 OpenAI SDK 客户端，但把 base_url 指到 OpenRouter 的 OpenAI 兼容网关
client=OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)


## 用 HTTP 直接调 OpenRouter

下一格不用 SDK 的 `chat.completions.create`，而是用 **`requests.post`** 向 OpenRouter 的  
`/api/v1/chat/completions` 发 JSON 请求体（含 `model`、`messages`、`reasoning`）。


In [34]:
# ========== 首次 API 调用：POST Chat Completions + 开启 reasoning ==========

# requests.post：同步 HTTP POST；url 指向 OpenRouter 的 chat completions 端点
response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    # Bearer Token：把环境变量里的 api_key 放进 Authorization
    "Authorization": f"Bearer {api_key}",
    # 告诉服务端请求体是 JSON
    "Content-Type": "application/json",
  },
  # data= 传已序列化的 JSON 字符串（不是 json= 字典参数）
  data=json.dumps({
    # 模型 id：必须与 OpenRouter 上可用的 slug 完全一致（勿改译）
    "model": "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    "messages": [
{
  "role": "system",
  # system prompt 保留英文：这是发给模型的指令，改译会改变回答风格/行为
  "content": """
You are a quantitative market analyst.

Your task:
- Interpret Shannon entropy and Gaussian entropy
- Analyze market structure
- Detect regime conditions
- Explain whether the market is:
  - trending
  - ranging
  - volatile
  - compressing
  - chaotic

Rules:
- NEVER reveal internal reasoning
- NEVER explain your thinking process
- ONLY provide the final market interpretation
- Keep response structured and professional
"""
},
      {
  "role": "user",
  # user：把前面算出的 shannon_entropy 与 G 嵌进 f-string，请模型做量化解读
  "content": f"""
Market entropy metrics:

Shannon Entropy: {shannon_entropy}

Gaussian Entropy: {G}

Interpret:
- structural randomness
- volatility randomness
- probable market regime
- trend strength
- market stability
- possible upcoming behavior

Respond like a professional quant analyst.
"""
}
      ],
    # reasoning.enabled：请求开启该模型的推理通道（具体行为由 OpenRouter/模型侧定义）
    "reasoning": {"enabled": True}
  })
)


In [35]:
# ========== 解析 JSON 响应并打印助手正文 ==========

# response.json()：把 HTTP 响应体解析成 Python 字典
response = response.json()
# OpenAI 兼容结构：choices[0].message.content 是助手最终文本
response = response['choices'][0]['message']['content']
# 打印市场解读（通常是结构化要点）
print(response)


**Market Interpretation**

- **Structural Randomness:** Moderate (Shannon entropy ≈ 2.75) – the market exhibits a blend of deterministic patterns and stochastic fluctuations, but not extreme chaos.  
- **Volatility Randomness:** Low (Gaussian entropy ≈ ‑10.24) – indicates compressed, low‑variance price movements, suggesting a regime of reduced volatility dispersion.  
- **Probable Market Regime:** Ranging/compressing – the combined entropy signals a stable, non‑trending environment where price action oscillates within a confined band.  
- **Trend Strength:** Weak – limited directional persistence is inferred from the entropy levels, implying any existing trend is fragile.  
- **Market Stability:** High – the negative Gaussian entropy points to a relatively stable price landscape with minimal abrupt swings.  
- **Possible Upcoming Behavior:** Expect continuation of range‑bound activity; a sustained move would require a rise in entropy (increased randomness) or a sharp expansion in volat

## 项目说明：基于熵的市场结构分析

该项目探索用 **Shannon 熵** 与 **Gaussian 熵** 作为定量工具，结合币安实时/近实时 K 线，观察金融市场结构与制度（regime）行为。

做法上不是只堆传统技术指标，而是把市场看成**信息动态系统**：随机性、波动性、结构组织可以用熵来度量。流程大致是：拉取历史蜡烛 → 算对数收益 → 估计收益分布的信息属性，从而判断市场更偏有组织、混乱、压缩、扩张、趋势还是震荡。

**Shannon 熵**：衡量结构不确定性。通过对收益直方图的概率分布求和，看价格行为更「有序」还是更「随机」。较低 Shannon 熵常对应更强结构/趋势痕迹；较高则更像噪声、犹豫、区间震荡。

**Gaussian 熵**：在连续收益近似正态时，由方差导出，反映波动离散程度。较高常对应波动扩张、不稳定；较低常对应波动压缩、安静积累。

二者结合可从「结构随机性 + 波动随机性」两个维度给市场状态分类，作为熵驱动量化研究的起点。实现上用到 Python、NumPy、pandas、requests、math 等；未来可扩展滚动熵、转移熵、政权切换模型等。


## 公式对照（本练习用到的两个熵）

- **Shannon 熵**（离散分布，手写实现）：  
  \(H(X)=-\sum_{i=1}^{n} p(x_i)\log_2 p(x_i)\)

- **Gaussian 熵**（连续正态近似，由方差算）：  
  \(H=\frac{1}{2}\log_2(2\pi e\,\sigma^2)\)
